In [1]:
import torch
import numpy as np
import pandas as pd
from datasets import load_dataset
from transformers import (
    BertTokenizer,
    BertForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)
from sklearn.metrics import accuracy_score, f1_score, classification_report
from torch.utils.data import Dataset

d:\Anaconda3\envs\NLP\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
GOEMOTIONS_MODEL_PATH = "../models/goemotions_bert"
OUTPUT_DIR             = "../models/dreaddit_stress_model"
NUM_LABELS             = 2
MAX_LEN                = 128
BATCH_SIZE             = 16
EPOCHS                 = 4
LEARNING_RATE          = 2e-5
SEED                   = 42

torch.manual_seed(SEED)

In [1]:
from datasets import load_dataset
import pandas as pd

print("Loading Dreaddit dataset...")

dataset = load_dataset(
    "csv",
    data_files={
        "train": "../dreaddit-train.csv",
        "test": "../dreaddit-test.csv"
    }
)

train_data = dataset["train"]
test_data  = dataset["test"]

print(f"Train samples : {len(train_data)}")
print(f"Test samples  : {len(test_data)}")
print(f"Label distribution (train): {pd.Series(train_data['label']).value_counts().to_dict()}")

d:\Anaconda3\envs\NLP\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading Dreaddit dataset...
Train samples : 2838
Test samples  : 715
Label distribution (train): {1: 1488, 0: 1350}


In [4]:
print(f"\nLoading GoEmotions model from: {GOEMOTIONS_MODEL_PATH}")

tokenizer = BertTokenizer.from_pretrained(GOEMOTIONS_MODEL_PATH)

# ignore_mismatched_sizes=True — replaces the old GoEmotions head (28 classes)
# with a new head (2 classes for Dreaddit), keeping all BERT weights
model = BertForSequenceClassification.from_pretrained(
    GOEMOTIONS_MODEL_PATH,
    num_labels=NUM_LABELS,
    ignore_mismatched_sizes=True   # KEY: swaps classification head only
)

print("Model loaded. Classification head replaced for 2-class stress output.")


Loading GoEmotions model from: ../models/goemotions_bert


Loading weights: 100%|██████████| 201/201 [00:00<?, ?it/s]
BertForSequenceClassification LOAD REPORT from: ../models/goemotions_bert
Key               | Status   |                                                                                        
------------------+----------+----------------------------------------------------------------------------------------
classifier.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([28, 768]) vs model:torch.Size([2, 768])
classifier.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([28]) vs model:torch.Size([2])          

Notes:
- MISMATCH	:ckpt weights were loaded, but they did not match the original empty weight shapes.


Model loaded. Classification head replaced for 2-class stress output.


In [5]:
def tokenize(batch):
    return tokenizer(
        batch["text"],
        padding="max_length",
        truncation=True,
        max_length=MAX_LEN
    )

print("\nTokenizing datasets...")
train_tokenized = train_data.map(tokenize, batched=True)
test_tokenized  = test_data.map(tokenize, batched=True)

# Rename label column to what Trainer expects
train_tokenized = train_tokenized.rename_column("label", "labels")
test_tokenized  = test_tokenized.rename_column("label", "labels")

# Set format for PyTorch
train_tokenized.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
test_tokenized.set_format("torch",  columns=["input_ids", "attention_mask", "labels"])


Tokenizing datasets...


In [6]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, predictions)
    f1  = f1_score(labels, predictions, average="weighted")
    return {"accuracy": acc, "f1": f1}

In [7]:
training_args = TrainingArguments(
    output_dir                  = OUTPUT_DIR,
    num_train_epochs            = EPOCHS,
    per_device_train_batch_size = BATCH_SIZE,
    per_device_eval_batch_size  = BATCH_SIZE,
    learning_rate               = LEARNING_RATE,
    warmup_ratio                = 0.1,           # warmup for 10% of steps
    weight_decay                = 0.01,
    eval_strategy               = "epoch",
    save_strategy               = "epoch",
    load_best_model_at_end      = True,
    metric_for_best_model       = "f1",
    logging_dir                 = "./logs",
    logging_steps               = 50,
    seed                        = SEED,
    fp16                        = torch.cuda.is_available(),  # faster if GPU available
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [8]:
trainer = Trainer(
    model           = model,
    args            = training_args,
    train_dataset   = train_tokenized,
    eval_dataset    = test_tokenized,
    compute_metrics = compute_metrics,
    callbacks       = [EarlyStoppingCallback(early_stopping_patience=2)]
)

In [9]:
print("\nStarting fine-tuning on Dreaddit...")
trainer.train()


Starting fine-tuning on Dreaddit...


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.429750,0.428650,0.800000,0.799341
2,0.296710,0.480275,0.795804,0.794124
3,0.155723,0.651588,0.798601,0.798471


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  3.72it/s]
There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer

TrainOutput(global_step=534, training_loss=0.3113561149840051, metrics={'train_runtime': 68.4682, 'train_samples_per_second': 165.8, 'train_steps_per_second': 10.399, 'total_flos': 560031881333760.0, 'train_loss': 0.3113561149840051, 'epoch': 3.0})

In [10]:
import torch

torch.save(model.state_dict(), 
           "D:/Projects/bert_mental_stress/models/dreaddit_final.pt")

print("Saved!")

Saved!
